<div style="background:linear-gradient(135deg,#143840 0%,#2B6264 100%);border-radius:14px;padding:32px 36px;color:#fff;font-family:'DM Sans',Arial,sans-serif;">
  <div style="font-size:11px;letter-spacing:2px;text-transform:uppercase;color:#FF4B31;font-weight:700;margin-bottom:10px;">Solutions Onboarding &middot; IBOR Training</div>
  <div style="font-size:30px;font-weight:700;line-height:1.15;margin-bottom:10px;">IBOR NB05 &mdash; Corporate Actions & Lifecycle Events</div>
  <div style="font-size:15px;color:rgba(255,255,255,.82);max-width:640px;line-height:1.55;">Apply the AAPL stock split via the corporate action source, and book the futures and options lifecycle events: in-the-money cash-settled exercises versus out-of-the-money expiries.</div>
</div>

<sub>IBOR pack sequence: NB00 &nbsp;&rarr;&nbsp; NB01 &nbsp;&rarr;&nbsp; NB02 &nbsp;&rarr;&nbsp; NB03 &nbsp;&rarr;&nbsp; NB04 &nbsp;&rarr;&nbsp; **NB05** &nbsp;&rarr;&nbsp; NB06 &nbsp;&rarr;&nbsp; NB07 &nbsp;&rarr;&nbsp; NB08</sub>

# NB05: Corporate Actions & Lifecycle Events

Demonstrates stock splits via legacy corporate actions, and verifies automatic lifecycle events (bond coupons, IRS cashflows, term deposit maturity).

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lusid-sdk', 'lumipy', '-q'],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
import os, json
from datetime import datetime, timedelta, timezone
import pandas as pd
import lusid as lu
import lusid.models as lm
import lumipy as lp
pd.set_option("display.max_columns", None)
SCOPE = 'ibor-training-v9'
QUOTE_SCOPE = 'ibor-training-v9-quotes'

def get_factory(app='lusid'):
    with open("secrets.json") as f: secrets = json.load(f)
    pat = secrets.get("api", {}).get("accessToken")
    module = __import__(app)
    if pat:
        return module.extensions.SyncApiClientFactory(config_loaders=[
            module.extensions.ArgsConfigurationLoader(api_url=secrets["api"]["lusidUrl"], access_token=pat)])
    return module.extensions.SyncApiClientFactory(config_loaders=[
        module.extensions.SecretsFileConfigurationLoader("secrets.json")])

def get_lumi():
    with open("secrets.json") as f: secrets = json.load(f)
    pat = secrets.get("api", {}).get("accessToken")
    if pat:
        return lp.get_client(access_token=pat, api_url=secrets["api"]["lusidUrl"].replace("/api", "/honeycomb"))
    return lp.get_client(api_secrets_file="secrets.json")

lusid_factory = get_factory()
lumi = get_lumi()
print('Ready')
transaction_portfolios_api = lusid_factory.build(lu.TransactionPortfoliosApi)
cas_api = lusid_factory.build(lu.CorporateActionSourcesApi)
instruments_api = lusid_factory.build(lu.InstrumentsApi)

---
## Stock Split: AAPL 2:1 on 2024-08-01

In [ ]:
# Get AAPL LUID
resp = instruments_api.get_instrument(identifier_type="ClientInternal", identifier="IBOR-AAPL", scope=SCOPE)
aapl_luid = resp.lusid_instrument_id
print(f"AAPL LUID: {aapl_luid}")

# Create the stock split event
try:
    cas_api.batch_upsert_corporate_actions(
        scope=SCOPE, code="IBOR-CA-SOURCE-V3",
        upsert_corporate_action_request=[
            lm.UpsertCorporateActionRequest(
                corporate_action_code="AAPL-SPLIT-2024",
                description="AAPL 2:1 stock split",
                announcement_date=datetime(2024, 7, 15, tzinfo=timezone.utc),
                ex_date=datetime(2024, 8, 1, tzinfo=timezone.utc),
                record_date=datetime(2024, 8, 1, tzinfo=timezone.utc),
                payment_date=datetime(2024, 8, 1, tzinfo=timezone.utc),
                transitions=[
                    lm.CorporateActionTransition(
                        input_transition=lm.CorporateActionTransitionComponent(
                            instrument_identifiers={"Instrument/default/LusidInstrumentId": aapl_luid},
                            instrument_scope=SCOPE,
                            instrument_uid=aapl_luid,
                            units_factor=1, cost_factor=1),
                        output_transitions=[
                            lm.CorporateActionTransitionComponent(
                                instrument_identifiers={"Instrument/default/LusidInstrumentId": aapl_luid},
                                instrument_scope=SCOPE,
                                instrument_uid=aapl_luid,
                                units_factor=2, cost_factor=1)]
                    )
                ]
            )
        ]
    )
    print("Created AAPL 2:1 stock split on 2024-08-01")
except lu.ApiException as e:
    if 'AlreadyExists' in str(e.body):
        print("Stock split already exists")
    else:
        print(f"Error: {str(e.body)[:300]}")

---
## Verify Stock Split

In [ ]:
print("AAPL holdings before and after split:")
for label, dt_str in [("Before (2024-07-31)", "2024-07-31"), ("After (2024-08-02)", "2024-08-02")]:
    dt = datetime.fromisoformat(dt_str).replace(tzinfo=timezone.utc)
    resp = transaction_portfolios_api.get_holdings(scope=SCOPE, code="IBOR-EQ", effective_at=dt.isoformat())
    for h in resp.values:
        if aapl_luid in str(h.instrument_uid):
            print(f"  {label}: {h.units:,.0f} units")

---
## Verify Bond Coupon Events (IBOR-FI)

In [ ]:
print("IBOR-FI USD cash over time (coupon income accumulating):")
for dt_str in ["2024-04-30", "2024-06-30", "2024-09-30", "2024-12-31"]:
    dt = datetime.fromisoformat(dt_str).replace(tzinfo=timezone.utc)
    resp = transaction_portfolios_api.get_holdings(scope=SCOPE, code="IBOR-FI", effective_at=dt.isoformat())
    cash = sum(h.units for h in resp.values if h.instrument_uid == 'CCY_USD')
    bonds = sum(1 for h in resp.values if h.holding_type == 'Position' and h.currency == 'USD')
    print(f"  {dt_str}: USD cash = {cash:,.2f}, bonds = {bonds}")

---
## Verify IRS Swap Cashflows (IBOR-MA)

In [ ]:
print("IBOR-MA system generated transactions (IRS cashflows + TD lifecycle):")
resp = transaction_portfolios_api.build_transactions(
    scope=SCOPE, code="IBOR-MA",
    transaction_query_parameters=lm.TransactionQueryParameters(
        start_date=datetime(2024, 6, 1, tzinfo=timezone.utc).isoformat(),
        end_date=datetime(2025, 6, 30, tzinfo=timezone.utc).isoformat()))

for t in resp.values:
    if t.type in ['SwapCashFlow', 'TermDepositInterest', 'TermDepositPrincipal', 'Maturity']:
        print(f"  [{t.type}] {t.transaction_date.date()}: {t.total_consideration.amount:,.2f} {t.total_consideration.currency}")

print("\nNB05 Complete. Proceed to NB06: Valuation & Recipe Configuration.")

---
## F&O Expiry and Settlement Events

Book cash settlement transactions for expired futures and exercised/expired options. ITM options are cash settled, OTM options expire worthless.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# F&O EXPIRY AND SETTLEMENT EVENTS
# Book cash settlement transactions for expired futures and options
# ═══════════════════════════════════════════════════════════════

print("Booking F&O expiry and settlement events...")

# Dec 2024 expirations (effective 2024-12-20)
dec_expiries = [
    # Futures: cash settle at final settlement price vs entry price
    # ESZ4: entered at ~5450, settled at 6050, 20 contracts * 50 multiplier
    {"ci": "IBOR-ESZ4", "type": "FutureCashSettlement", "units": 20, "price": 6050,
     "date": "2024-12-20", "pnl": (6050 - 5450) * 50 * 20},
    
    # Options: ITM exercise as cash settlement, OTM expire worthless
    # AAPL C200 Deep ITM: AAPL ~242, intrinsic = 42, 10 contracts * 100 shares
    {"ci": "IBOR-AAPL-C200", "type": "CashSettledOptionExercise", "units": 10, "price": 42.00,
     "date": "2024-12-20", "pnl": 42.00 * 100 * 10},
    # AMZN P220 ITM: AMZN ~195, intrinsic = 25, 15 contracts * 100 shares
    {"ci": "IBOR-AMZN-P220", "type": "CashSettledOptionExercise", "units": 15, "price": 25.00,
     "date": "2024-12-20", "pnl": 25.00 * 100 * 15},
    # GOOGL C190 ATM: GOOGL ~193, intrinsic = 3.50, 10 contracts * 100 shares
    {"ci": "IBOR-GOOGL-C190", "type": "CashSettledOptionExercise", "units": 10, "price": 3.50,
     "date": "2024-12-20", "pnl": 3.50 * 100 * 10},
]

# Dec 2024 Treasury future: physical delivery, just remove the position
dec_expiries.append(
    {"ci": "IBOR-TYZ4", "type": "FutureCashSettlement", "units": 10, "price": 112.25,
     "date": "2024-12-19", "pnl": (112.25 - 110.50) * 1000 * 10}
)

# Book all Dec expirations
txn_requests = []
for exp in dec_expiries:
    txn_requests.append(lm.TransactionRequest(
        transaction_id=f"EXPIRY-{exp['ci']}-DEC24",
        type=exp['type'],
        instrument_identifiers={"Instrument/default/ClientInternal": exp['ci']},
        transaction_date=datetime.strptime(exp['date'], "%Y-%m-%d").replace(tzinfo=timezone.utc).isoformat(),
        settlement_date=datetime.strptime(exp['date'], "%Y-%m-%d").replace(tzinfo=timezone.utc).isoformat(),
        units=exp['units'],
        transaction_price=lm.TransactionPrice(price=exp['price'], type="Price"),
        total_consideration=lm.CurrencyAndAmount(amount=exp['pnl'], currency="USD"),
        transaction_currency="USD"
    ))
    print(f"  {exp['type']}: {exp['ci']} | {exp['units']} units @ {exp['price']} | P&L: ${exp['pnl']:,.0f}")

transaction_portfolios_api.upsert_transactions(scope=SCOPE, code="IBOR-MA", transaction_request=txn_requests)
print(f"  Booked {len(txn_requests)} Dec 2024 expiry transactions")

# Jan 2025: WTI Crude Oil future
jan_expiries = [
    {"ci": "IBOR-CLF5", "type": "FutureCashSettlement", "units": 5, "price": 72.50,
     "date": "2025-01-21", "pnl": (72.50 - 78.50) * 1000 * 5},
]

txn_requests_jan = []
for exp in jan_expiries:
    txn_requests_jan.append(lm.TransactionRequest(
        transaction_id=f"EXPIRY-{exp['ci']}-JAN25",
        type=exp['type'],
        instrument_identifiers={"Instrument/default/ClientInternal": exp['ci']},
        transaction_date=datetime.strptime(exp['date'], "%Y-%m-%d").replace(tzinfo=timezone.utc).isoformat(),
        settlement_date=datetime.strptime(exp['date'], "%Y-%m-%d").replace(tzinfo=timezone.utc).isoformat(),
        units=exp['units'],
        transaction_price=lm.TransactionPrice(price=exp['price'], type="Price"),
        total_consideration=lm.CurrencyAndAmount(amount=exp['pnl'], currency="USD"),
        transaction_currency="USD"
    ))
    print(f"  {exp['type']}: {exp['ci']} | {exp['units']} units @ {exp['price']} | P&L: ${exp['pnl']:,.0f}")

transaction_portfolios_api.upsert_transactions(scope=SCOPE, code="IBOR-MA", transaction_request=txn_requests_jan)
print(f"  Booked {len(txn_requests_jan)} Jan 2025 expiry transactions")

# Mar 2025: MSFT Put (OTM expires worthless), NVDA Call (ITM), META Call (OTM worthless), NQ future, Gold future
mar_expiries = [
    # MSFT P380 OTM: expires worthless
    {"ci": "IBOR-MSFT-P380", "type": "Expiry", "units": 10, "price": 0,
     "date": "2025-03-21", "pnl": 0},
    # NVDA C150 Deep ITM: NVDA ~215, intrinsic = 65
    {"ci": "IBOR-NVDA-C150", "type": "CashSettledOptionExercise", "units": 5, "price": 65.00,
     "date": "2025-03-21", "pnl": 65.00 * 100 * 5},
    # META C600 OTM: expires worthless
    {"ci": "IBOR-META-C600", "type": "Expiry", "units": 20, "price": 0,
     "date": "2025-03-21", "pnl": 0},
    # NQ future
    {"ci": "IBOR-NQH5", "type": "FutureCashSettlement", "units": 10, "price": 21500,
     "date": "2025-03-21", "pnl": (21500 - 18800) * 20 * 10},
]

# Feb 2025: Gold future
feb_expiries = [
    {"ci": "IBOR-GCG5", "type": "FutureCashSettlement", "units": 5, "price": 2650,
     "date": "2025-02-26", "pnl": (2650 - 2050) * 100 * 5},
]

for month_label, expiries in [("Feb 2025", feb_expiries), ("Mar 2025", mar_expiries)]:
    txn_batch = []
    for exp in expiries:
        month_code = month_label.replace(" ", "").upper()[:5]
        txn_batch.append(lm.TransactionRequest(
            transaction_id=f"EXPIRY-{exp['ci']}-{month_code}",
            type=exp['type'],
            instrument_identifiers={"Instrument/default/ClientInternal": exp['ci']},
            transaction_date=datetime.strptime(exp['date'], "%Y-%m-%d").replace(tzinfo=timezone.utc).isoformat(),
            settlement_date=datetime.strptime(exp['date'], "%Y-%m-%d").replace(tzinfo=timezone.utc).isoformat(),
            units=exp['units'],
            transaction_price=lm.TransactionPrice(price=exp['price'], type="Price"),
            total_consideration=lm.CurrencyAndAmount(amount=exp['pnl'], currency="USD"),
            transaction_currency="USD"
        ))
        status = "EXERCISED" if exp['type'] == 'CashSettledOptionExercise' else ("EXPIRED WORTHLESS" if exp['pnl'] == 0 else "SETTLED")
        print(f"  {status}: {exp['ci']} | P&L: ${exp['pnl']:,.0f}")
    
    transaction_portfolios_api.upsert_transactions(scope=SCOPE, code="IBOR-MA", transaction_request=txn_batch)
    print(f"  Booked {len(txn_batch)} {month_label} transactions")

print("\nF&O lifecycle events complete.")
print("  Verify: View IBOR-MA holdings at 2025-01-01 vs 2025-04-01")
print("  Dec expiries should be gone by Jan, Mar expiries gone by Apr")
print("  Cash balance should reflect settlement P&L")